# Trực quan hóa đúng pipeline 65-view multi-scale TTA

Notebook này dùng trực tiếp `build_specs`, `apply_center_zoom`, `upscale_small_image` và `unwrap_plate_lines` từ `benchmark_multiscale_tta.py`, cùng các cấu hình trong `preprocessing.py`. Vì vậy ảnh minh họa không phải mockup: chúng là kết quả của implementation hiện tại.

Cấu trúc đúng là **1 baseline + 40 full + 24 unwrap = 65 views**. Không tồn tại một `two-line baseline` riêng. Unwrap là phép biến đổi có điều kiện bên trong 24 view: nếu `width / height >= 1.9`, hàm unwrap trả nguyên ảnh đầu vào.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image as DisplayImage, display
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == 'preprocessing_best_config':
    ROOT = ROOT.parent
CONFIG_DIR = ROOT / 'preprocessing_best_config'
if str(CONFIG_DIR) not in sys.path:
    sys.path.insert(0, str(CONFIG_DIR))

from benchmark_multiscale_tta import build_specs
from visualize_multiview_pipeline import export_visualization

specs = build_specs()
full_specs = [s for s in specs if s.name.startswith('full_')]
unwrap_specs = [s for s in specs if s.name.startswith('unwrap_')]
assert (len(specs), len(full_specs), len(unwrap_specs)) == (65, 40, 24)
print(f'Đã kiểm tra spec: 1 baseline + {len(full_specs)} full + {len(unwrap_specs)} unwrap = {len(specs)}')

## Chọn crop minh họa

Mặc định dùng một biển hai dòng thật trong repository. `DEMO_SIZE=(96, 70)` mô phỏng crop độ phân giải thấp để cả nhánh upscale 2× và 3× thực sự được kích hoạt theo đúng điều kiện trong code (`width < 128` hoặc `height < 64`). Đặt `DEMO_SIZE=None` nếu muốn giữ nguyên kích thước ảnh.

In [ ]:
SOURCE = ROOT / 'wrong_images' / '59DB05813.png'
OUTPUT_DIR = CONFIG_DIR / 'multiview_pipeline_images'
DEMO_SIZE = (96, 70)

display(Image.open(SOURCE))
print('Source:', SOURCE)
print('Output:', OUTPUT_DIR)

## Sinh toàn bộ ảnh

Cell này ghi 65 model-input view riêng lẻ, manifest CSV/JSON, các contact sheet, ảnh minh họa unwrap và pipeline cuối vào `OUTPUT_DIR`.

In [ ]:
summary = export_visualization(
    source_path=SOURCE,
    output_dir=OUTPUT_DIR,
    demo_size=DEMO_SIZE,
)
summary

## Kiểm tra khác biệt hình học và tiền xử lý

Ảnh đầu cho thấy crop hai dòng được tách tại vùng có năng lượng nét dọc thấp rồi ghép `dòng trên | dòng dưới`. Các sheet kế tiếp cho thấy 4 preprocessor và toàn bộ 65 view, có nhãn để không nhầm các biến thể.

In [ ]:
for name in [
    '02_unwrap_geometry.png',
    '05_four_preprocessors.png',
    '03_full_40_views.png',
    '04_unwrap_24_views.png',
    '06_all_65_views.png',
]:
    print(name)
    display(DisplayImage(filename=str(OUTPUT_DIR / name)))

## Pipeline cuối

Sơ đồ cuối được dựng deterministic từ chính các PNG đã xuất ở trên. Nội dung sửa hai lỗi của hình cũ: bỏ `two-line view` giả ở cạnh baseline, và thể hiện rõ unwrap là nhánh 24 view có ảnh hình học khác full view.

In [ ]:
FINAL_PIPELINE = OUTPUT_DIR / 'multiview_65_pipeline.png'
display(DisplayImage(filename=str(FINAL_PIPELINE)))
print('Saved:', FINAL_PIPELINE)